In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AlkenylReduction(MorphingOperator):
    def __init__(self):
        super(AlkenylReduction, self).__init__()
        self._name = "Alkenyl Reduction (Phase I - Ring Safe)"
        self._target_bonds = []
        self.PATTERN = Chem.MolFromSmarts("[CX3;!a]=[CX3;!a]")

    def setOriginal(self, mol):
        super(AlkenylReduction, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.PATTERN is not None:
            matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
            for match in matches:
                pair = tuple(sorted([match[0], match[1]]))
                if pair not in self._target_bonds:
                    self._target_bonds.append(pair)

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
        
        idx1, idx2 = random.choice(self._target_bonds)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
                bond.SetStereo(Chem.BondStereo.STEREONONE)
            
            for idx in [idx1, idx2]:
                atom = rw_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
                if atom.HasProp('_CIPCode'): 
                    atom.ClearProp('_CIPCode')
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)
            
            new_mol = rw_mol.GetMol()
            
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

alkene_op = AlkenylReduction()

print("=== STARTING ALKENYL REDUCTION CRASH-TEST ===")
# 1. Έλεγχος Προστασίας Αρωματικότητας
benzene = MolpherMol("C1=CC=CC=C1")
alkene_op.setOriginal(benzene)
prod_benzene = alkene_op.morph()
print(f"Βενζόλιο (Αρωματικό):\n  SRC: {benzene.getSMILES()}\n  TRG: {prod_benzene.getSMILES()} (Safe)")

print("\n=== SIMULATING TWO-GENERATION MORPHIC TREE ===")
root_smiles = "CC=CC.C1=CCCCC1"
gen0_mol = MolpherMol(root_smiles)
print(f"GENERATION 0 (Root):\n  SMILES: {gen0_mol.getSMILES()}")

# Γενιά 1: Πρώτη αναγωγή (θα διαλέξει τυχαία έναν από τους δύο δεσμούς)
alkene_op.setOriginal(gen0_mol)
gen1_mol = alkene_op.morph()
print(f"GENERATION 1:\n  SMILES: {gen1_mol.getSMILES()}")

# Γενιά 2: Δεύτερη αναγωγή (θα κορεστεί και ο εναπομείνων δεσμός)
alkene_op.setOriginal(gen1_mol)
gen2_mol = alkene_op.morph()
print(f"GENERATION 2 (Fully Saturated):\n  SMILES: {gen2_mol.getSMILES()}")
print("==============================================")

=== STARTING ALKENYL REDUCTION CRASH-TEST ===
Βενζόλιο (Αρωματικό):
  SRC: C1=CC=CC=C1
  TRG: C1=CC=CC=C1 (Safe)

=== SIMULATING TWO-GENERATION MORPHIC TREE ===
GENERATION 0 (Root):
  SMILES: C1=CCCCC1.CC=CC
GENERATION 1:
  SMILES: C1CCCCC1.CC=CC
GENERATION 2 (Fully Saturated):
  SMILES: C1CCCCC1.CCCC
